# EmotionCLIP-ReID RAF-DB Runbook

Notebook này dành riêng cho RAF-DB Basic. Protocol dùng official predefined split của RAF-DB: `train_*` để train và `test_*` để validation/test trong pipeline hiện tại. Không random split.

## 0. Kernel và repo

Chạy cell này trong đúng kernel/environment bạn muốn train.

In [7]:
from pathlib import Path
import os
import sys

REPO = Path.cwd()
if not (REPO / "train_emotionclip.py").exists():
    candidates = [Path("/mnt/e/Source/EmotionCLIP-ReID"), Path.home() / "EmotionCLIP-ReID"]
    for candidate in candidates:
        if (candidate / "train_emotionclip.py").exists():
            REPO = candidate
            os.chdir(REPO)
            break
print("Repo:", REPO)
print("Python:", sys.executable)

Repo: /home/jupyter-hault/EmotionCLIP-ReID
Python: /opt/tljh/user/envs/py310/bin/python


## 1. Kiểm tra package tối thiểu

In [8]:
import importlib.util
import subprocess
import sys

required = ["torch", "PIL", "yaml", "numpy","kaggle"]
missing = [name for name in required if importlib.util.find_spec(name) is None]
print("Missing:", missing)
if missing:
    print("Cài environment theo environment_emotionclip_cuda.yml hoặc cài các package còn thiếu trước khi train.")

# Kaggle chỉ cần nếu bạn muốn tải bằng Kaggle thay vì upload archive RAF-DB sẵn.
print("Kaggle module:", importlib.util.find_spec("kaggle") is not None)

Missing: []
Kaggle module: True


## 1.1. Cài Kaggle downloader

Chạy cell này nếu muốn tải RAF-DB qua Kaggle. Sau khi cài, cần cấu hình `~/.kaggle/kaggle.json` hoặc set env `KAGGLE_USERNAME` và `KAGGLE_KEY`. Nếu dùng archive local qua `RAF_ARCHIVE` thì có thể bỏ qua credential Kaggle.


In [9]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

for package in ["kagglehub", "kaggle"]:
    if importlib.util.find_spec(package) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)
        print(f"Installed {package} package")
    else:
        print(f"{package} package already installed")

kaggle_json = Path.home() / ".kaggle" / "kaggle.json"
access_token = Path.home() / ".kaggle" / "access_token"
has_legacy_env_auth = bool(os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"))
has_token_env_auth = bool(os.environ.get("KAGGLE_API_TOKEN"))

print("~/.kaggle/access_token exists:", access_token.exists())
print("~/.kaggle/kaggle.json exists:", kaggle_json.exists())
print("KAGGLE_API_TOKEN env exists:", has_token_env_auth)
print("KAGGLE_USERNAME/KAGGLE_KEY env exists:", has_legacy_env_auth)

if access_token.exists():
    access_token.chmod(0o600)
    print("Set ~/.kaggle/access_token permission to 600")
if kaggle_json.exists():
    kaggle_json.chmod(0o600)
    print("Set ~/.kaggle/kaggle.json permission to 600")
if not access_token.exists() and not kaggle_json.exists() and not has_token_env_auth and not has_legacy_env_auth:
    print("Chưa có Kaggle credential. Với token mới, tạo ~/.kaggle/access_token hoặc set KAGGLE_API_TOKEN.")


Installed kagglehub package
kaggle package already installed
~/.kaggle/access_token exists: False
~/.kaggle/kaggle.json exists: False
KAGGLE_API_TOKEN env exists: False
KAGGLE_USERNAME/KAGGLE_KEY env exists: False
Chưa có Kaggle credential. Với token mới, tạo ~/.kaggle/access_token hoặc set KAGGLE_API_TOKEN.


## 2. Tải và process RAF-DB

Có 2 cách dùng cell này:

- Upload archive RAF-DB chính thức và set `RAF_ARCHIVE`.
- Hoặc cấu hình Kaggle credential rồi tải dataset bằng `KAGGLE_DATASET`.

Converter sẽ tạo `manifest.jsonl` theo official split, không random split.

In [10]:
from pathlib import Path
import importlib.util
import os
import shutil
import subprocess
import sys

RAF_ROOT = REPO / "data" / "RAF-DB"
RAF_MANIFEST = RAF_ROOT / "manifest.jsonl"
RAF_ARCHIVE = None  # Ví dụ: "/home/jovyan/data/RAF-DB.zip"
KAGGLE_DATASET = "shuvoalok/raf-db-dataset"

if RAF_ARCHIVE:
    print(f"Using local RAF-DB archive: {RAF_ARCHIVE}")
    convert_cmd = [
        sys.executable,
        "tools/convert_rafdb_to_emotion_jsonl.py",
        "--archive",
        str(RAF_ARCHIVE),
        "--extract-dir",
        str(RAF_ROOT),
        "--output",
        str(RAF_MANIFEST),
        "--root-dir",
        str(RAF_ROOT),
    ]
else:
    kaggle_json = Path.home() / ".kaggle" / "kaggle.json"
    access_token = Path.home() / ".kaggle" / "access_token"
    has_legacy_env_auth = bool(os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"))
    has_token_env_auth = bool(os.environ.get("KAGGLE_API_TOKEN"))
    has_kagglehub_auth = access_token.exists() or has_token_env_auth
    has_legacy_auth = kaggle_json.exists() or has_legacy_env_auth

    if not has_kagglehub_auth and not has_legacy_auth:
        raise RuntimeError(
            "Không thấy Kaggle credential. Với token mới, tạo ~/.kaggle/access_token hoặc set KAGGLE_API_TOKEN. "
            "Với legacy API, cấu hình ~/.kaggle/kaggle.json hoặc KAGGLE_USERNAME + KAGGLE_KEY."
        )

    RAF_ROOT.mkdir(parents=True, exist_ok=True)
    if has_kagglehub_auth:
        if importlib.util.find_spec("kagglehub") is None:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub"], check=True)
        import kagglehub

        if access_token.exists():
            access_token.chmod(0o600)
        print(f"Downloading Kaggle dataset with kagglehub: {KAGGLE_DATASET} -> {RAF_ROOT}")
        downloaded_path = Path(kagglehub.dataset_download(KAGGLE_DATASET, output_dir=str(RAF_ROOT)))
        print("Downloaded path:", downloaded_path)
    else:
        if importlib.util.find_spec("kaggle") is None:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)
        if kaggle_json.exists():
            kaggle_json.chmod(0o600)
        print(f"Downloading Kaggle dataset with legacy kaggle CLI: {KAGGLE_DATASET} -> {RAF_ROOT}")
        subprocess.run(
            [
                sys.executable,
                "-m",
                "kaggle",
                "datasets",
                "download",
                "-d",
                KAGGLE_DATASET,
                "-p",
                str(RAF_ROOT),
                "--unzip",
            ],
            check=True,
        )

    convert_cmd = [
        sys.executable,
        "tools/convert_rafdb_to_emotion_jsonl.py",
        "--raf-root",
        str(RAF_ROOT),
        "--output",
        str(RAF_MANIFEST),
        "--root-dir",
        str(RAF_ROOT),
    ]

print("Converting RAF-DB official split -> EmotionCLIP manifest")
subprocess.run(convert_cmd, cwd=REPO, check=True)
print("RAF-DB manifest ready:", RAF_MANIFEST)


RuntimeError: Không thấy Kaggle credential. Với token mới, tạo ~/.kaggle/access_token hoặc set KAGGLE_API_TOKEN. Với legacy API, cấu hình ~/.kaggle/kaggle.json hoặc KAGGLE_USERNAME + KAGGLE_KEY.

## 3. Kiểm tra manifest và phân bố split

In [ ]:
import json
from collections import Counter

assert RAF_MANIFEST.exists(), f"Missing manifest: {RAF_MANIFEST}"
records = [json.loads(line) for line in RAF_MANIFEST.read_text(encoding="utf-8").splitlines() if line.strip()]
print("Total records:", len(records))
print("Split counts:", Counter(record["split"] for record in records))
print("Official split counts:", Counter(record.get("official_split") for record in records))
print("Emotion counts:", Counter(record["emotion"] for record in records))
print("Split protocol:", records[0].get("split_protocol"))
records[:3]

## 4. Preview ảnh mẫu

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

sample_records = records[:8]
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, record in zip(axes.ravel(), sample_records):
    image_path = RAF_ROOT / record["image_path"]
    ax.imshow(Image.open(image_path).convert("RGB"))
    ax.set_title(f"{record['split']} / {record['emotion']}")
    ax.axis("off")
plt.tight_layout()

## 5. Train RAF-DB

In [ ]:
import subprocess
import sys

RAF_CONFIG = REPO / "configs" / "emotion" / "vit_b16_emotionclip_rafdb_quick.yml"
OUTPUT_DIR = REPO / "outputs" / "emotionclip_rafdb_quick"

train_cmd = [
    sys.executable,
    "train_emotionclip.py",
    "--config_file",
    str(RAF_CONFIG),
    "DATASETS.MANIFEST",
    str(RAF_MANIFEST),
    "DATASETS.ROOT_DIR",
    str(RAF_ROOT),
    "OUTPUT_DIR",
    str(OUTPUT_DIR),
]
subprocess.run(train_cmd, cwd=REPO, check=True)

## 6. Xem metrics tốt nhất hoặc epoch mới nhất

In [ ]:
import json
from pathlib import Path

metric_files = sorted(OUTPUT_DIR.glob("metrics_epoch_*.json"))
print("Metric files:", [path.name for path in metric_files[-5:]])
if metric_files:
    metrics = json.loads(metric_files[-1].read_text(encoding="utf-8"))
    for key in ["accuracy", "balanced_accuracy", "macro_f1", "avg_uncertainty", "avg_confidence", "ece", "uncertainty_risk_auc", "num_samples"]:
        print(key, metrics.get(key))
    print("Per-class F1:", metrics.get("per_class_f1"))

## 7. Infer một ảnh validation

In [ ]:
import subprocess
import sys

val_records = [record for record in records if record["split"] == "val"]
sample = val_records[0]
weight = OUTPUT_DIR / "best_emotionclip.pth"
image_path = RAF_ROOT / sample["image_path"]
print("Ground truth:", sample["emotion"])
print("Image:", image_path)
subprocess.run(
    [
        sys.executable,
        "infer_emotionclip.py",
        "--config_file",
        str(RAF_CONFIG),
        "--weight",
        str(weight),
        "--image",
        str(image_path),
    ],
    cwd=REPO,
    check=True,
)